# 2.4 — Vector Stores

A **vector store** is a database that stores embeddings and lets you search them by similarity.

We'll use **Chroma** — a lightweight, local vector database that runs in Python.

```
Documents
   ↓ embed
Vectors  →  Store in Chroma  →  Query by similarity  →  Return top-k chunks
```

In [6]:
!pip install langchain langchain-core langchain-ollama langchain-community chromadb --quiet

## 1. Prepare Documents

In [7]:
from langchain_core.documents import Document

docs = [
    Document(page_content='All full-time employees receive 20 days of annual leave per year.', metadata={'section': 'Leave Policy'}),
    Document(page_content='Sick leave is up to 10 days per year with a medical certificate.', metadata={'section': 'Leave Policy'}),
    Document(page_content='Parental leave is 16 weeks fully paid for primary caregivers.', metadata={'section': 'Leave Policy'}),
    Document(page_content='Employees may work remotely up to 3 days per week.', metadata={'section': 'Remote Work'}),
    Document(page_content='Remote workers must be available during core hours: 10am to 3pm.', metadata={'section': 'Remote Work'}),
    Document(page_content='All remote work equipment is provided by the company.', metadata={'section': 'Remote Work'}),
    Document(page_content='Health insurance is provided for all full-time employees and their immediate family.', metadata={'section': 'Benefits'}),
    Document(page_content='A gym membership subsidy of $50 per month is available.', metadata={'section': 'Benefits'}),
    Document(page_content='Employees receive a $1,000 annual learning and development budget.', metadata={'section': 'Benefits'}),
    Document(page_content='Standard working hours are 9am to 5pm, Monday to Friday.', metadata={'section': 'Working Hours'}),
    Document(page_content='Overtime must be pre-approved and will be compensated at 1.5x the hourly rate.', metadata={'section': 'Working Hours'}),
]

print(f'Prepared {len(docs)} documents')

Prepared 11 documents


## 2. Create a Chroma Vector Store

In [14]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='llama3.1')

# Create vector store from documents
# This embeds all documents and stores them in Chroma
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory='./chroma_db'  # save to disk
)

print(f'Vector store created with {vectorstore._collection.count()} documents')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store created with 22 documents


## 3. Similarity Search

In [19]:
query = 'How many vacation days do I get?'
results = vectorstore.similarity_search(query, k=7)

print(f'Query: "{query}"\n')
for i, doc in enumerate(results):
    print(f'Result {i+1} [{doc.metadata["section"]}]:')
    print(f'  {doc.page_content}')
    print()

Query: "How many vacation days do I get?"

Result 1 [Benefits]:
  Employees receive a $1,000 annual learning and development budget.

Result 2 [Benefits]:
  Employees receive a $1,000 annual learning and development budget.

Result 3 [Remote Work]:
  Employees may work remotely up to 3 days per week.

Result 4 [Remote Work]:
  Employees may work remotely up to 3 days per week.

Result 5 [Benefits]:
  A gym membership subsidy of $50 per month is available.

Result 6 [Benefits]:
  A gym membership subsidy of $50 per month is available.

Result 7 [Leave Policy]:
  All full-time employees receive 20 days of annual leave per year.



## 4. Similarity Search with Scores

In [11]:
query = 'Can I work from home?'
results_with_scores = vectorstore.similarity_search_with_score(query, k=4)

print(f'Query: "{query}"\n')
for doc, score in results_with_scores:
    print(f'Score: {score:.4f}  [{doc.metadata["section"]}]')
    print(f'  {doc.page_content}')
    print()

Query: "Can I work from home?"

Score: 0.6206  [Remote Work]
  Employees may work remotely up to 3 days per week.

Score: 0.6719  [Remote Work]
  All remote work equipment is provided by the company.

Score: 0.7159  [Remote Work]
  Remote workers must be available during core hours: 10am to 3pm.

Score: 0.7194  [Benefits]
  Employees receive a $1,000 annual learning and development budget.



## 5. Use as a Retriever

A retriever wraps the vector store and integrates into LangChain pipelines.

In [16]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

results = retriever.invoke('What benefits does the company offer?')

print('Retrieved documents:')
for doc in results:
    print(f'  [{doc.metadata["section"]}] {doc.page_content}')

Retrieved documents:
  [Remote Work] Employees may work remotely up to 3 days per week.
  [Remote Work] Employees may work remotely up to 3 days per week.
  [Benefits] Employees receive a $1,000 annual learning and development budget.


## 6. Load an Existing Vector Store from Disk

In [17]:
# Load the persisted store — no need to re-embed!
loaded_store = Chroma(
    persist_directory='./chroma_db',
    embedding_function=embeddings
)

results = loaded_store.similarity_search('overtime pay', k=2)
for doc in results:
    print(f'[{doc.metadata["section"]}] {doc.page_content}')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


[Working Hours] Overtime must be pre-approved and will be compensated at 1.5x the hourly rate.
[Working Hours] Overtime must be pre-approved and will be compensated at 1.5x the hourly rate.


## Summary

| Concept | Description |
|---------|-------------|
| `Chroma.from_documents()` | Create a vector store from a list of documents |
| `similarity_search()` | Find top-k most similar documents to a query |
| `similarity_search_with_score()` | Same but also returns similarity scores |
| `as_retriever()` | Wrap the vector store for use in LangChain chains |
| `persist_directory` | Save the vector store to disk to avoid re-embedding |